# Notebook de Validacion — AutoML P2 (Paralelismo + Backtesting Temporal)

Este notebook reproduce, en Google Colab, el mismo resultado que correr:

```
python main.py --input data/input --config config/config.yaml --parallel
```

NO es el entregable principal (ese es el codigo modular en `src/`); es evidencia
de que el pipeline corre igual en otro entorno. Corre el flujo completo:
preprocesamiento -> HPO -> entrenamiento (paralelo, con benchmark de speed-up)
-> GetBestModel() (criterio P1) -> revalidacion con backtesting temporal (P2).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Copia (o clona) la carpeta del proyecto a Colab y ajusta la ruta de abajo para
que apunte a la raiz de `automl_venta_financiera/`. A diferencia de la P1, los
CSV de las particiones van **directo** en `data/input/` (ya no en
`data/input/data/`).

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/MLOps/automl_venta_financiera'  # <-- editar
DATASET_PATH = f'{PROJECT_ROOT}/data/input'  # <-- coloca aqui los CSV p1_extrac.csv...pN_extrac.csv

import sys, os
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


In [ ]:
!pip install -q catboost lightgbm PyYAML


## 1. Cargar configuracion y preprocesar

In [ ]:
import yaml
from src import preprocessing, training, parallelism, evaluation, registry
from src.hpo import buscar_hiperparametros

with open('config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['dataset']['input_path'] = DATASET_PATH

# Si todavia no tienes todas las particiones descargadas, ajusta el cutoff
# para dejar al menos 4 particiones futuras disponibles para el backtesting.
# config['backtesting']['train_cutoff_partition'] = 2

pre_data = preprocessing.preprocesar(config)
print('Particiones usadas para entrenar (hasta el cutoff):', pre_data['cutoff_aplicado'])
print('Meses ordenados (p_fecinformacion):', pre_data['meses_ordenados'])
print('Mes de validacion (P1) usado:', pre_data['mes_validacion_usado'])


## 2. Optimizacion de hiperparametros (HPO)

In [ ]:
hpo_results = buscar_hiperparametros(pre_data, config)
for nombre, info in hpo_results.items():
    print(f"{nombre}: best_cv_score={info['best_cv_score']}  best_params={info['best_params']}")


## 3. Entrenamiento paralelo (con benchmark de speed-up)

Se mide el mismo conjunto de experimentos en un for-loop secuencial
(`T_secuencial`) y con `joblib.Parallel` (`T_paralelo`); el resultado que se
usa para el resto del pipeline es el de la corrida paralela.

In [ ]:
experimentos = training.construir_lista_experimentos(config, hpo_results)
resultados, info_parallelism = parallelism.ejecutar_con_benchmark(experimentos, pre_data, config)

for r in resultados:
    r['metadata']['parallelism'] = info_parallelism

print(info_parallelism)


## 4. Seleccion del campeon: criterio P1 + revalidacion con backtesting (P2)

In [ ]:
seleccion = evaluation.GetBestModel(
    resultados,
    max_decay_pct=config['selection']['max_decay_pct'],
    usar_criterio_innovador=config['selection']['usar_criterio_innovador'],
)

revalidacion = evaluation.revalidar_con_backtesting(seleccion, pre_data, config)
campeon = revalidacion['campeon_final']

print('Ganador inicial (criterio P1):', seleccion['ganador_final']['metadata']['ml_name'])
print('Campeon final (tras backtesting):', campeon['metadata']['ml_name'])
print('Motivo de cambio:', revalidacion['motivo_cambio'])


## 5. Tabla comparativa completa

In [ ]:
import pandas as pd

tabla = pd.DataFrame(training.construir_tabla_resultados(seleccion['ranking_completo']))
tabla.sort_values('auc_test', ascending=False)


## 6. Resultados de backtesting del campeon (por ola)

In [ ]:
bt = campeon['metadata']['backtesting']
tabla_bt = pd.DataFrame(bt['waves'])
print('AUC train:', campeon['metadata']['performance']['auc_train'])
print('AUC promedio backtesting:', bt['avg_auc_backtest'], '| decay promedio:', bt['avg_decay_backtest'], '%')
print('Alerta de drift:', bt['drift_alert'], '(umbral:', bt['drift_alert_threshold'], '%)')
tabla_bt


## 7. Guardar la corrida (mismos archivos que genera `main.py`)

In [ ]:
carpeta_corrida = registry.crear_carpeta_corrida(config['output']['output_dir'])

for r in resultados:
    registry.guardar_modelo_y_metadata(
        carpeta_corrida, r['metadata']['ml_name'], r['modelo'], r['metadata']
    )

registry.guardar_backtesting_results(carpeta_corrida, campeon['metadata']['backtesting'])

registry.guardar_resumen_corrida(carpeta_corrida, {
    'ganador_base': seleccion['ganador_base']['metadata'],
    'ganador_innovador': seleccion['ganador_innovador']['metadata'],
    'ganador_final_criterio_p1': seleccion['ganador_final']['metadata'],
    'campeon_final_tras_backtesting': campeon['metadata'],
    'motivo_cambio_backtesting': revalidacion['motivo_cambio'],
    'candidatos_probados_en_backtesting': revalidacion['candidatos_probados'],
    'criterio_usado': seleccion['criterio_usado'],
    'parallelism': info_parallelism,
    'hpo_results': hpo_results,
    'cutoff_aplicado': pre_data['cutoff_aplicado'],
    'meses_ordenados': [int(m) for m in pre_data['meses_ordenados']],
    'mes_validacion_usado': pre_data['mes_validacion_usado'],
    'ranking_completo': [r['metadata'] for r in seleccion['ranking_completo']],
})

print('Corrida guardada en:', carpeta_corrida)
